# RQ1: Apache Kafka (Standalone, Self-Contained)

Real extraction, matching, and code mining for Apache Kafka, the second independent replication project. Output: `kafka_real_mined_dataset.csv`.

In [1]:
!pip install -q pandas numpy requests lizard || pip install -q pandas numpy requests lizard --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.0 MB/s eta 0:00:00


In [2]:
"""
RQ1: Apache Kafka
================================================================================

"""

import subprocess
import os
import re
import time
import requests
import pandas as pd
import numpy as np
from collections import defaultdict

HEADERS = {"User-Agent": "qm640-capstone"}
np.random.seed(42)


# ---------------------------------------------------------------------------
# Step 1: Real JIRA extraction for Kafka (unchanged, not the slow part)
# ---------------------------------------------------------------------------
def extract_kafka_jira_issues():
    JIRA_BASE_URL = "https://issues.apache.org/jira/rest/api/2/search"
    PAGE_SIZE = 100
    all_issues = []
    start_at = 0
    jql = 'project=KAFKA AND resolution=Fixed ORDER BY resolutiondate ASC'
    fields = "created,resolutiondate,priority,components,comment,summary,status,issuetype"

    while True:
        params = {"jql": jql, "startAt": start_at, "maxResults": PAGE_SIZE, "fields": fields}
        resp = requests.get(JIRA_BASE_URL, params=params, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        issues = data.get("issues", [])
        if not issues:
            break
        for issue in issues:
            f = issue["fields"]
            all_issues.append({
                "issue_id": issue["key"],
                "project_name": "KAFKA",
                "created": f.get("created"),
                "resolution_date": f.get("resolutiondate"),
                "priority": (f.get("priority") or {}).get("name"),
                "component": ", ".join(c["name"] for c in f.get("components", [])),
                "num_comments": (f.get("comment") or {}).get("total", 0),
                "issue_type": (f.get("issuetype") or {}).get("name"),
            })
        start_at += PAGE_SIZE
        print(f"  KAFKA: fetched {len(all_issues)} real issues so far...")
        time.sleep(0.5)
        if start_at >= data.get("total", 0):
            break

    df = pd.DataFrame(all_issues)
    df.to_csv("kafka_jira_raw.csv", index=False)
    print(f"Real Kafka JIRA extraction complete: {len(df)} real issues")
    return df


def ensure_kafka_repo():
    path = "repos/kafka"
    if not os.path.exists(path):
        print("Cloning real Apache Kafka repository (full history, blobless)...")
        subprocess.run(["git", "clone", "--filter=blob:none", "https://github.com/apache/kafka.git", path], check=True)
    else:
        print(f"Kafka repository already present at {path}")
    return path


# ---------------------------------------------------------------------------
#real indexed commit lookup (one pass, not one per issue)
# ---------------------------------------------------------------------------
def build_issue_to_commit_index(repo_dir: str) -> dict:
    print("Reading full real Kafka commit history (one pass, this is the slow-but-only-once step)...")
    result = subprocess.run(
        ["git", "-C", repo_dir, "log", "--all", "--format=%H|%s"],
        capture_output=True, text=True, timeout=300
    )
    index = defaultdict(list)
    pattern = re.compile(r"KAFKA-\d+", re.IGNORECASE)
    for line in result.stdout.splitlines():
        if "|" not in line:
            continue
        sha, message = line.split("|", 1)
        for issue_id in pattern.findall(message):
            index[issue_id.upper()].append(sha)
    print(f"  Real index built: {len(index)} unique real Kafka issue IDs found in commit history")
    return index


def get_changed_java_files(repo_dir, sha):
    result = subprocess.run(
        ["git", "-C", repo_dir, "diff-tree", "--no-commit-id", "--name-only", "-r", sha],
        capture_output=True, text=True, timeout=30
    )
    files = [f for f in result.stdout.splitlines() if f.endswith((".java", ".scala"))]
    return [f for f in files if "/test/" not in f and "/generated/" not in f]


# ---------------------------------------------------------------------------
# FIX for Bug 2: real batch file reader (one process, not one per file)
# ---------------------------------------------------------------------------
class BatchFileReader:
    """FIXED: same real bug found and fixed for Camel/Hadoop applied here --
    text-mode pipe + early return without consuming the response body could
    desync the stream and hang/crash on a non-blob (tree/commit) object."""

    def __init__(self, repo_dir: str):
        self.proc = subprocess.Popen(
            ["git", "-C", repo_dir, "cat-file", "--batch"],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE
        )

    def read(self, sha: str, path: str) -> str:
        self.proc.stdin.write(f"{sha}:{path}\n".encode("utf-8"))
        self.proc.stdin.flush()
        header = self.proc.stdout.readline().decode("utf-8", errors="replace")
        parts = header.split()
        if len(parts) < 2 or parts[1] == "missing":
            return ""
        try:
            size = int(parts[2])
        except (ValueError, IndexError):
            return ""
        content_bytes = self.proc.stdout.read(size)
        self.proc.stdout.read(1)
        if parts[1] != "blob":
            return ""
        return content_bytes.decode("utf-8", errors="replace")

    def close(self):
        self.proc.stdin.close()
        self.proc.wait()


def analyze_commit_metrics(reader: "BatchFileReader", sha: str, files: list):
    import lizard
    total_nloc, total_complexity_sum, total_functions = 0, 0, 0
    files_analyzed = 0
    for path in files:
        content = reader.read(sha, path)
        if not content.strip():
            continue
        try:
            analysis = lizard.analyze_file.analyze_source_code(path, content)
        except Exception:
            continue
        total_nloc += analysis.nloc
        total_functions += len(analysis.function_list)
        for fn in analysis.function_list:
            total_complexity_sum += fn.cyclomatic_complexity
        files_analyzed += 1
    if files_analyzed == 0 or total_functions == 0:
        return None
    return {
        "loc": total_nloc,
        "cyclomatic_complexity": total_complexity_sum / total_functions,
        "num_functions": total_functions,
        "num_files_changed": files_analyzed,
    }


def run_kafka_extension(sample_per_era: int = 300, save_every: int = 25):
    jira_df = extract_kafka_jira_issues()
    jira_df["resolution_date_parsed"] = pd.to_datetime(jira_df["resolution_date"], errors="coerce", utc=True)
    jira_df["era"] = (jira_df["resolution_date_parsed"] >= "2023-01-01").map({True: "ai_era", False: "pre_ai"})

    repo_dir = ensure_kafka_repo()

    # FIXED: build the real index ONCE instead of a per-issue git log --grep
    commit_index = build_issue_to_commit_index(repo_dir)
    matched_ids = set(jira_df["issue_id"]) & set(commit_index.keys())
    print(f"Real match rate: {len(matched_ids)} of {len(jira_df)} "
          f"({len(matched_ids)/len(jira_df)*100:.1f}%) Kafka issues have a real matching commit")

    matchable = jira_df[jira_df["issue_id"].isin(matched_ids)]

    samples = []
    for era in ["pre_ai", "ai_era"]:
        cell = matchable[matchable.era == era]
        n = min(sample_per_era, len(cell))
        samples.append(cell.sample(n=n, random_state=42))
        print(f"Real {era} matchable pool: {len(cell)}, sampling {n}")
    sample_df = pd.concat(samples, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

    # FIXED: one persistent batch reader instead of a new subprocess per file
    reader = BatchFileReader(repo_dir)

    rows = []
    for i, (_, row) in enumerate(sample_df.iterrows()):
        shas = commit_index.get(row["issue_id"].upper(), [])  # instant lookup
        if not shas:
            continue
        sha = shas[0]
        files = get_changed_java_files(repo_dir, sha)
        if not files:
            continue
        metrics = analyze_commit_metrics(reader, sha, files)
        if metrics is None:
            continue
        metrics.update({
            "issue_id": row["issue_id"], "project_name": "KAFKA", "era": row["era"],
            "issue_type": row["issue_type"], "defect_prone": 1 if row["issue_type"] == "Bug" else 0,
        })
        rows.append(metrics)
        if (i + 1) % save_every == 0:
            pd.DataFrame(rows).to_csv("kafka_mining_progress.csv", index=False)
            print(f"  [{i+1}/{len(sample_df)}] processed, {len(rows)} real samples so far")

    reader.close()

    out = pd.DataFrame(rows)
    out.to_csv("kafka_real_mined_dataset.csv", index=False)
    print(f"\nKAFKA EXTENSION COMPLETE: {len(out)} real mined code samples")
    return out


if __name__ == "__main__":
    result = run_kafka_extension(sample_per_era=300)


  KAFKA: fetched 100 real issues so far...
  KAFKA: fetched 100 real issues so far...
  KAFKA: fetched 200 real issues so far...
  KAFKA: fetched 200 real issues so far...
  KAFKA: fetched 300 real issues so far...
  KAFKA: fetched 300 real issues so far...
  KAFKA: fetched 400 real issues so far...
  KAFKA: fetched 400 real issues so far...
  KAFKA: fetched 500 real issues so far...
  KAFKA: fetched 500 real issues so far...
  KAFKA: fetched 600 real issues so far...
  KAFKA: fetched 600 real issues so far...
  KAFKA: fetched 700 real issues so far...
  KAFKA: fetched 700 real issues so far...
  KAFKA: fetched 800 real issues so far...
  KAFKA: fetched 800 real issues so far...
  KAFKA: fetched 900 real issues so far...
  KAFKA: fetched 900 real issues so far...
  KAFKA: fetched 1000 real issues so far...
  KAFKA: fetched 1000 real issues so far...
  KAFKA: fetched 1100 real issues so far...
  KAFKA: fetched 1100 real issues so far...
  KAFKA: fetched 1200 real issues so far...
  KAFK